# EDA inicial

Este notebook replica el análisis exploratorio del dataset *horseshoe crab* siguiendo la línea de Agresti (2002), complementado con una exploración de la abundancia de ceros.

El foco es la relación entre el número de satélites (`Sa`) y el ancho del caparazón de la hembra (`W`, en cm). La secuencia del análisis es:

1. Visualizar la distribución de `Sa` y su relación con `W`
2. Identificar indicios de sobredispersión
3. Ajustar un modelo base con distribución Poisson
4. Ajustar un modelo Binomial Negativo y comparar los resultados
5. Explorar la abundancia de ceros y las limitaciones de ambos modelos ante este problema

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## Análisis exploratorio de datos

In [ ]:
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
display(df.head())
print(df.shape)

El punto de partida es la Figura 4.3 de Agresti (2002): un diagrama de dispersión de `Sa` contra `W`. Este gráfico permite identificar si existe una tendencia positiva entre el tamaño de la hembra y el número de satélites que atrae.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['W'], df['Sa'], alpha=0.6, color='gray')
plt.title('Número de satélites según ancho del caparazón')
plt.xlabel('Ancho del caparazón (W, cm)')
plt.ylabel('Número de satélites (Sa)')
plt.tight_layout()
plt.show()

Dado que `Sa` es una variable de conteo discreta y los valores de `W` están concentrados en un rango estrecho, el diagrama de dispersión genera sobreposición de puntos (*overplotting*) que oculta la acumulación en ciertas regiones.

El gráfico hexagonal resuelve este problema agrupando observaciones en celdas hexagonales y codificando su frecuencia por color. Aun así, la tendencia global no es inmediatamente evidente — hay demasiada variabilidad vertical en `Sa` para cada valor de `W`.

In [ ]:
plt.figure(figsize=(8, 5))
hb = plt.hexbin(df['W'], df['Sa'], gridsize=20, cmap='viridis_r', mincnt=1)
plt.colorbar(hb, label='Número de observaciones')
plt.title('Distribución conjunta: satélites y ancho del caparazón')
plt.xlabel('Ancho del caparazón (W, cm)')
plt.ylabel('Número de satélites (Sa)')
plt.tight_layout()
plt.show()

Para visualizar la tendencia con mayor claridad, utilizamos dos herramientas complementarias: una regresión no paramétrica con suavizamiento LOWESS (*Locally Weighted Scatterplot Smoothing*) y las medias de `Sa` calculadas en bins de `W`.

Discretizamos `W` en ocho intervalos y calculamos, por bin, el número de observaciones, la media y la varianza de `Sa`, y la proporción de ceros. La curva LOWESS se superpone como referencia de la tendencia global.

In [ ]:
# Suavizamiento LOWESS sobre los datos originales
smooth_series = pd.DataFrame(lowess(df['Sa'], df['W'], frac=0.5), columns=['W', 'Sa'])

# Discretizar W en 8 bins y calcular estadísticos por bin
df['W_cat'] = pd.cut(
    df['W'],
    bins=[-1, 23.25, 24.25, 25.25, 26.25, 27.25, 28.25, 29.25, np.inf],
    labels=['<23.25', '23.25-24.25', '24.25-25.25', '25.25-26.25',
            '26.25-27.25', '27.25-28.25', '28.25-29.25', '>29.25']
)
df['flag_is_zero'] = (df['Sa'] == 0)

cat_summary = df.groupby('W_cat').agg(
    n_obs=('W_cat', 'count'),
    sum_satellites=('Sa', 'sum'),
    mean_satellites=('Sa', 'mean'),
    var_satellites=('Sa', 'var'),
    mean_w=('W', 'mean'),
    prop_zeros=('flag_is_zero', 'mean')
).round(2).reset_index()

display(cat_summary)

# Medias por bin + curva LOWESS
plt.figure(figsize=(8, 5))
sns.scatterplot(x='mean_w', y='mean_satellites', data=cat_summary,
                color='darkcyan', label='Medias por bin')
sns.lineplot(x='W', y='Sa', data=smooth_series,
             label='Suavizado LOWESS', linestyle='--', color='black')
plt.title('Tendencia de satélites por ancho del caparazón')
plt.xlabel('Ancho del caparazón (W, cm)')
plt.ylabel('Número de satélites (Sa)')
plt.ylim(0, 5.5)
plt.tight_layout()
plt.show()

A partir de la tabla anterior, graficamos la media y la varianza de `Sa` por bin. Bajo el supuesto Poisson, la varianza condicional debe ser igual a la media: $\text{Var}(Y \mid X) = \mathbb{E}[Y \mid X]$. En los datos, la varianza es sistemáticamente al menos 3 veces mayor que la media — un diagnóstico claro de **sobredispersión**.

In [ ]:
plot_df = cat_summary[['W_cat', 'mean_satellites', 'var_satellites']].melt(
    id_vars='W_cat', var_name='Estadístico', value_name='valor'
)
plot_df['Estadístico'] = plot_df['Estadístico'].map({
    'mean_satellites': 'Media',
    'var_satellites': 'Varianza'
})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    x='W_cat', y='valor', hue='Estadístico', data=plot_df,
    palette=['steelblue', 'tomato'], ax=ax
)
ax.set_xlabel('Bin de ancho (cm)')
ax.set_ylabel('Satélites')
ax.set_title('Media vs Varianza de satélites por bin de ancho')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Modelo Poisson

Ajustamos un GLM con distribución Poisson y liga logarítmica. Sabemos de antemano que fallará en dispersión, pero es el punto de partida necesario: establecer la línea base y cuantificar exactamente dónde falla.

Estandarizamos `W` en `W_scaled` para facilitar la interpretación de los coeficientes. Usamos la API de fórmulas de `statsmodels`, cuya sintaxis y estructura de output son similares a las de `R`.

In [ ]:
# Estandarizar W para mejorar la interpretabilidad de los coeficientes
mean_W = df['W'].mean()
std_W  = df['W'].std()
df['W_scaled'] = (df['W'] - mean_W) / std_W

print(f'Media W:           {mean_W:.4f} cm')
print(f'Desv. estándar W:  {std_W:.4f} cm')

# Ajuste del GLM Poisson con liga logarítmica
model = smf.poisson(formula='Sa ~ W_scaled', data=df)
results = model.fit(disp=0)
print(results.summary())

Los resultados indican que la ecuación de regresión ajustada es:

$$
\log(\hat{\mu}) = 1.0094 + 0.346\, W_{\text{scaled}}
$$

o bien, expresada en función de `W` original:

$$
\log(\hat{\mu}) = 1.0094 + 0.346\, \left( \frac{W - 26.30}{2.11} \right).
$$

Con el modelo ajustado podemos calcular $\hat{\mu}$ para nuevos valores de `W`.

In [ ]:
# Predicciones para dos valores representativos de W
new_W1 = 26.3  # cerca de la media (26.30 cm)
new_W2 = 28.4  # cerca de la media + 1 SD (28.41 cm)
new_W1_scaled = (new_W1 - mean_W) / std_W
new_W2_scaled = (new_W2 - mean_W) / std_W

poisson_preds_new_W = results.predict(pd.DataFrame({'W_scaled': [new_W1_scaled, new_W2_scaled]}))
print(poisson_preds_new_W)

### Modelo Binomial Negativo para sobredispersión

La distribución Binomial Negativa agrega un parámetro $\phi$ que desacopla la media de la varianza:

$$\text{Var}(Y) = \mu + \frac{\mu^2}{\phi} > \mu.$$

Conforme $\phi \to \infty$, el modelo converge a Poisson. Un valor pequeño de $\hat{\phi}$ indica sobredispersión marcada. Ajustamos el modelo con la misma fórmula para comparar directamente los coeficientes.

In [ ]:
# Ajuste del GLM Binomial Negativo con la misma fórmula
model_nb = smf.negativebinomial(formula='Sa ~ W_scaled', data=df)
results_nb = model_nb.fit(disp=0)
print(results_nb.summary())

## Comparación de modelos

La ecuación de regresión del Binomial Negativo es prácticamente idéntica a la de Poisson:

$$
\log(\hat{\mu}) = 0.9988 + 0.4051\, W_{\text{scaled}}.
$$

Las predicciones puntuales de $\hat{\mu}$ tampoco difieren de forma sustancial. La diferencia real está en la estimación de la varianza.

In [ ]:
# Las predicciones puntuales son similares a las de Poisson;
# la diferencia relevante está en los intervalos de confianza
nb_preds_new_W = results_nb.predict(pd.DataFrame({'W_scaled': [new_W1_scaled, new_W2_scaled]}))
print(nb_preds_new_W)

La diferencia entre modelos se manifiesta en la incertidumbre de las predicciones. A primera vista puede parecer que Poisson es *mejor* porque sus intervalos son más estrechos — en realidad está **subestimando el error**. El Binomial Negativo, al estimar $\phi$ desde los datos, produce intervalos que reflejan la dispersión real.

In [ ]:
x_range = np.linspace(df['W_scaled'].min(), df['W_scaled'].max(), 100)
df_plot = pd.DataFrame({'W_scaled': x_range})
x_range_W = x_range * std_W + mean_W  # convertir a escala original para el eje x

pred_poisson = results.get_prediction(df_plot).summary_frame(alpha=0.05)
pred_nb      = results_nb.get_prediction(df_plot).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(df['W'], df['Sa'], alpha=0.5, color='gray', label='Observaciones')

# Curva Poisson con intervalo de confianza al 95%
plt.plot(x_range_W, pred_poisson['predicted'], color='steelblue', lw=2, label='Poisson')
plt.fill_between(x_range_W, pred_poisson['ci_lower'], pred_poisson['ci_upper'],
                 color='steelblue', alpha=0.15, label='IC 95% Poisson')

# Curva Binomial Negativo con intervalo de confianza al 95%
plt.plot(x_range_W, pred_nb['predicted'], color='tomato', lw=2, label='Binomial Negativo')
plt.fill_between(x_range_W, pred_nb['ci_lower'], pred_nb['ci_upper'],
                 color='tomato', alpha=0.15, label='IC 95% Bin. Negativo')

plt.title('Predicciones: Poisson vs Binomial Negativo')
plt.xlabel('Ancho del caparazón (W, cm)')
plt.ylabel('Número de satélites (Sa)')
plt.legend()
plt.tight_layout()
plt.show()

Para contrastar formalmente los modelos usamos la **prueba de razón de verosimilitud** (*Likelihood Ratio Test*, LRT). Bajo $H_0: \phi \to \infty$ (Poisson como caso particular del Binomial Negativo), el estadístico $2\left(\ell_{\text{NB}} - \ell_{\text{P}}\right)$ se distribuye asintóticamente como $\chi^2_1$.

Un p-value cercano a cero indica evidencia suficiente para rechazar el modelo Poisson en favor del Binomial Negativo.

In [ ]:
# LRT: 2*(LL_NB - LL_Poisson) ~ chi²(df=1) bajo H0: phi -> inf (Poisson)
ll_poisson = results.llf
ll_nb       = results_nb.llf
lr_stat     = 2 * (ll_nb - ll_poisson)
p_value     = stats.chi2.sf(lr_stat, df=1)

print(f'Estadístico LRT:            {lr_stat:.4f}')
print(f'Distribución de referencia: χ²(df=1)')
print(f'p-value:                    {p_value:.2e}')

## Abundancia de ceros

Alrededor del 33% de las hembras en el dataset no tienen ningún satélite. Evaluamos si los modelos ajustados reproducen adecuadamente esta característica de los datos.

Comparamos la proporción observada de ceros por bin de `W` contra la probabilidad de observar un cero según cada modelo. Para Poisson, $P(Y=0 \mid \mu) = e^{-\mu}$; para Binomial Negativo, $P(Y=0 \mid \mu, \phi) = \left(\frac{\phi}{\phi + \mu}\right)^\phi$.

Ambos modelos subestiman la proporción de ceros observada, especialmente en los bins de `W` intermedio. Esta discrepancia sugiere que los ceros en los datos no se explican únicamente por la distribución de conteo base — existe un proceso adicional que los genera, lo que motiva los modelos *Zero-Inflated* que exploraremos en notebooks posteriores.

In [ ]:
prop_zeros_obs = (df['Sa'] == 0).mean()  # proporción global observada de ceros

plt.figure(figsize=(8, 5))

# Proporción observada por bin
sns.scatterplot(x='mean_w', y='prop_zeros', data=cat_summary,
                color='black', s=60, zorder=3, label='Observado (por bin)')

# P(Y=0) predicha según cada modelo
sns.lineplot(x=x_range_W, y=results.predict(df_plot, which='prob')[0],
             label='Poisson', color='steelblue', lw=2)
sns.lineplot(x=x_range_W, y=results_nb.predict(df_plot, which='prob')[0],
             label='Binomial Negativo', color='tomato', lw=2)

# Línea de referencia: proporción global observada
plt.axhline(prop_zeros_obs, color='black', linestyle='--', lw=1,
            label=f'Media observada ({prop_zeros_obs:.1%})')

plt.title('Proporción de ceros observada vs predicha')
plt.xlabel('Ancho del caparazón (W, cm)')
plt.ylabel('Proporción de ceros P(Y = 0)')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()